# BAL Score Changes, Teacher Comparison, and Regional Matched Student Counts for Cambridge TKT-Based Professional Development Program Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This dataset describes teacher-level summaries of matched student English BAL summative assessment marks for secondary-school teachers in Azerbaijan who participated in an online Cambridge TKT-based professional development programme.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

In this notebook, we work from the Croissant JSON-LD URL and use variables so the workflow stays reusable.

In [ ]:
import json
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.f74k-xed8/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

In [ ]:
# Convert metadata to JSON for inspection when needed
metadata_json = metadata.to_json()
print(json.dumps({
    '@id': metadata_json.get('@id'),
    'name': metadata_json.get('name'),
    'identifier': metadata_json.get('identifier'),
    'temporalCoverage': metadata_json.get('temporalCoverage'),
    'spatialCoverage': metadata_json.get('spatialCoverage')
}, indent=2))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Per the instructions, all dataset entities should be referenced by their `@id` values. We first inspect the Croissant metadata directly to see whether record sets are defined.

In [ ]:
record_sets_metadata = metadata_json.get('recordSet', [])
print(f"Number of record sets declared in metadata: {len(record_sets_metadata)}")

if record_sets_metadata:
    for rs in record_sets_metadata:
        print('-' * 80)
        print(f"Record set @id: {rs.get('@id')}")
        print(f"Name: {rs.get('name', 'N/A')}")
        fields = rs.get('field', [])
        print(f"Field count: {len(fields)}")
        for field in fields:
            print(f"  Field @id: {field.get('@id')} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
else:
    print('No record sets are declared in the Croissant metadata for this dataset.')

Because this dataset package currently exposes no `recordSet` entries, there are no tabular records to load with `dataset.records(record_set=...)`.

Instead, we will explore the dataset through its dataset-level Croissant metadata fields, which still contain rich information for analysis.

In [ ]:
# Show key top-level metadata fields and their values
top_level_fields = [
    '@id', 'name', 'description', 'identifier', 'citeAs', 'datePublished',
    'version', 'license', 'spatialCoverage', 'temporalCoverage',
    'keywords', 'dataCollectionType', 'dataUseCases', 'dataBiases',
    'dataLimitations', 'dataPreprocessingProtocol'
]

for field_id in top_level_fields:
    value = metadata_json.get(field_id, None)
    print(f"{field_id}: {value}\n")

## 3. Data Extraction
Load structured metadata elements into pandas DataFrames for analysis.

Since no record sets are available, we create DataFrames from list-valued dataset fields, using their top-level metadata keys as identifiers.

In [ ]:
# Extract list-valued metadata fields into DataFrames
record_sets = []  # No record sets available in this Croissant package

list_field_ids = [
    'keywords',
    'dataCollectionType',
    'dataUseCases',
    'dataBiases',
    'dataLimitations',
    'dataPreprocessingProtocol',
    'dataCollectionTimeframe',
    'dataReleaseMaintenancePlan',
    'personalSensitiveInformation'
]

dataframes = {}

for field_id in list_field_ids:
    values = metadata_json.get(field_id, [])
    if isinstance(values, list):
        dataframes[field_id] = pd.DataFrame({field_id: values})
    else:
        dataframes[field_id] = pd.DataFrame({field_id: [values]})

print('Created DataFrames:')
print(list(dataframes.keys()))

In [ ]:
# Inspect one extracted metadata DataFrame
example_record_set_id = 'dataUseCases'
print(dataframes[example_record_set_id].columns.tolist())
dataframes[example_record_set_id].head()

We can also assemble a compact one-row summary table from scalar metadata fields to make the dataset easier to inspect programmatically.

In [ ]:
summary_field_ids = [
    '@id', 'name', 'identifier', 'datePublished', 'version',
    'license', 'spatialCoverage', 'temporalCoverage', 'description'
]

summary_row = {field_id: metadata_json.get(field_id) for field_id in summary_field_ids}
dataset_summary_df = pd.DataFrame([summary_row])
dataset_summary_df

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps to the extracted metadata tables.

Because the Croissant package does not expose tabular record sets, this EDA section focuses on text- and count-based exploration of metadata lists. We will:

- filter entries by length,
- normalize text lengths,
- summarize grouped counts,
- and inspect protocol complexity.

In [ ]:
# Select a metadata-derived field for analysis
record_set_id = 'dataLimitations'
numeric_field = 'text_length'
text_field_id = 'dataLimitations'

limitations_df = dataframes[record_set_id].copy()
limitations_df[numeric_field] = limitations_df[text_field_id].str.len()

threshold = 90
filtered_df = limitations_df[limitations_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
    filtered_df[numeric_field].std(ddof=0)
    if len(filtered_df) > 1 and filtered_df[numeric_field].std(ddof=0) != 0
    else 0
)

print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

group_field = 'starts_with'
filtered_df[group_field] = filtered_df[text_field_id].str.split().str[0]
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name='mean_text_length')
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())

In [ ]:
# Explore preprocessing protocol steps by word count
protocol_record_set_id = 'dataPreprocessingProtocol'
protocol_field_id = 'dataPreprocessingProtocol'

protocol_df = dataframes[protocol_record_set_id].copy()
protocol_df['word_count'] = protocol_df[protocol_field_id].str.split().str.len()
protocol_df['step_id'] = ['step_' + str(i + 1) for i in range(len(protocol_df))]

protocol_df.sort_values('word_count', ascending=False)

In [ ]:
# Compare the sizes of list-based metadata sections
section_sizes = pd.DataFrame({
    'section_@id': list(dataframes.keys()),
    'item_count': [len(df) for df in dataframes.values()]
}).sort_values('item_count', ascending=False)

section_sizes

## 5. Visualization
Visualize distributions and counts derived from the dataset metadata.

In [ ]:
# Visualization 1: Number of items in each metadata section
plt.figure(figsize=(10, 5))
sns.barplot(data=section_sizes, x='item_count', y='section_@id', palette='viridis')
plt.title('Counts of Items Across Metadata Sections')
plt.xlabel('Item count')
plt.ylabel('Metadata section @id')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 2: Distribution of data limitation text lengths
plt.figure(figsize=(8, 5))
sns.histplot(limitations_df['text_length'], bins=8, kde=True, color='steelblue')
plt.title('Distribution of Text Lengths in dataLimitations')
plt.xlabel('Character count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Visualization 3: Word counts for preprocessing steps
plt.figure(figsize=(10, 5))
sns.barplot(data=protocol_df, x='step_id', y='word_count', palette='magma')
plt.title('Word Count by dataPreprocessingProtocol Step')
plt.xlabel('Protocol step @id surrogate')
plt.ylabel('Word count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Conclusion
This exploration shows that the Croissant package contains rich dataset-level metadata but currently declares no `recordSet` entities, so direct tabular record loading through `dataset.records(record_set=...)` is not available.

Key observations:
- The dataset is focused on teacher-level aggregated outcomes in Azerbaijan.
- The metadata includes detailed documentation of collection methods, preprocessing, limitations, and intended use cases.
- The package is well suited for metadata inspection, documentation analysis, and responsible data-use review.
- If future versions add record sets and field `@id` definitions, the same notebook structure can be extended to load and analyze row-level or table-level records directly with `mlcroissant`.